# 03 - Tool calling and structured output

Two related but distinct ideas live in this notebook:

1. **Tool calling**: the model decides, on its own, that it wants to run a function you defined. You execute the function and feed the result back. Repeat until the model is done. This is the core of every agent.
2. **Structured output**: you do not want free-form text back, you want a typed object with specific fields. Both ideas use the same provider feature under the hood (function/tool calling), so they sit next to each other naturally.

By the end you will have written the manual tool-calling loop by hand. That matters because in notebook 4 we use `create_agent`, which automates this exact loop. Knowing what it does saves you a lot of guessing later.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model
model = init_chat_model("openai:gpt-4o-mini")

## Defining a tool: the `@tool` decorator

A tool is a Python function the model is allowed to call. Wrap any function with `@tool` and LangChain reads three things off it:

1. The function name → tool name shown to the model
2. The docstring → tool description shown to the model
3. The type hints → JSON schema of the arguments

Those three pieces together are what gets sent to the provider. The provider then decides when (and with what arguments) to call your tool. This is why you should write a *good* docstring and use *real* type hints. They are not for humans, they are part of the prompt.

In [ ]:
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city. The city should be a real place name."""
    fake_db = {"boston": "42F, drizzle", "tokyo": "68F, clear", "london": "55F, cloudy"}
    return fake_db.get(city.lower(), f"No data for {city}.")

@tool
def add(a: float, b: float) -> float:
    """Add two numbers and return the result."""
    return a + b

print("name:        ", get_weather.name)
print("description: ", get_weather.description)
print("args schema: ", get_weather.args)

Notice that `get_weather` is no longer a plain function. It has been wrapped into a `Tool` object that exposes `.name`, `.description`, `.args`, and supports the same `invoke` interface every other Runnable does. You can call `get_weather.invoke({"city": "Boston"})` directly and it works like a normal function.

## `bind_tools`: tell the model what is available

By itself, the model has no idea your tools exist. You attach them with `bind_tools`. The result is a *new* model object (the original is untouched) that knows about those tools.

In [ ]:
tools = [get_weather, add]
model_with_tools = model.bind_tools(tools)

response = model_with_tools.invoke("What's the weather in Boston right now?")

print("text content:", repr(response.text))
print("tool_calls:  ", response.tool_calls)

### What just happened

Look at the output carefully. The `text` is empty (or near-empty), but `tool_calls` has an entry. The model is saying: "I am not answering you yet. I want you to run `get_weather` with `{'city': 'Boston'}` first."

**This is the most important thing to internalize about tool calling**: a tool call is not the model calling your function. The model only emits *intent*. Your code is the one that actually runs the tool and feeds the result back. The model is just talking; your code is doing.

Each entry in `tool_calls` has:

- `name`: which tool the model wants
- `args`: the arguments it picked
- `id`: a unique id you must echo back when you return the result

## The manual tool-calling loop

Here it is, written out the long way. This is the same loop `create_agent` runs internally. Walk through every line.

In [ ]:
from langchain.messages import HumanMessage, ToolMessage

# index our tools by name so we can look them up by what the model picks
tools_by_name = {t.name: t for t in tools}

messages = [HumanMessage("What's the weather in Boston?")]

# 1) ask the model
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)
print("AI requested tool calls:", ai_msg.tool_calls)

# 2) for each tool the model wanted, run it and append the result as a ToolMessage
for tool_call in ai_msg.tool_calls:
    tool_obj = tools_by_name[tool_call["name"]]
    result = tool_obj.invoke(tool_call["args"])
    messages.append(ToolMessage(content=str(result), tool_call_id=tool_call["id"]))

# 3) ask the model again, this time it has the tool results in context
final = model_with_tools.invoke(messages)
print()
print("final answer:", final.text)

### Insight: the four message types from notebook 1, in action

Look at the `messages` list at the end of that loop. It has:

1. `HumanMessage` (the user's question)
2. `AIMessage` (with `tool_calls`, no real text)
3. `ToolMessage` (the result of `get_weather`, tagged with `tool_call_id` so the model knows which call it answers)
4. `AIMessage` (the final answer)

Now go back and re-read the table in notebook 1. The four message types you saw there are exactly the four pieces of an agent turn. Tools are the only reason `ToolMessage` exists at all.

In [ ]:
# Look at the full message sequence we built
for m in messages:
    print(f"{type(m).__name__:>15} | text={m.text!r}")

## A loop that handles multiple rounds

What if the question requires *more than one* tool call in sequence? "What's the weather in Boston, and add 5 to the temperature in Fahrenheit" might have the model call `get_weather` first, see the result, then call `add`. The single-step loop above will not handle that.

Generalize: keep looping until the model returns a response with no `tool_calls`.

In [ ]:
def run_agent_loop(user_input: str, max_steps: int = 6):
    messages = [HumanMessage(user_input)]
    for step in range(max_steps):
        ai_msg = model_with_tools.invoke(messages)
        messages.append(ai_msg)
        if not ai_msg.tool_calls:
            print(f"[step {step}] done")
            return ai_msg.text, messages
        print(f"[step {step}] calling: {[tc['name'] for tc in ai_msg.tool_calls]}")
        for tool_call in ai_msg.tool_calls:
            tool_obj = tools_by_name[tool_call["name"]]
            result = tool_obj.invoke(tool_call["args"])
            messages.append(ToolMessage(content=str(result), tool_call_id=tool_call["id"]))
    return "hit max_steps without finishing", messages

answer, history = run_agent_loop("What is 17 plus 25? Then what is the weather in Tokyo?")
print("\nFINAL:", answer)

**Insight**: that 30-line function is, for practical purposes, what an agent is. Add some retries, tracing, persistence, and a graph for branching control flow, and you have `create_agent` (which is built on LangGraph). The framework adds robustness, but the *core idea* fits in one screen. If you ever forget what an agent is doing, come back to this loop.

## Parallel tool calls

Most modern providers will emit multiple tool calls in a single `AIMessage` if the user asked for several things at once. Our loop already handles this (the `for tool_call in ai_msg.tool_calls` line iterates over all of them). Let's see it happen:

In [ ]:
msg = model_with_tools.invoke("What's the weather in Boston AND in Tokyo?")
for tc in msg.tool_calls:
    print(tc["name"], tc["args"])

If a particular call should never be parallel (e.g. a stateful action like "send email"), you can disable it: `model.bind_tools([t], parallel_tool_calls=False)`.

## Forcing a tool call

Sometimes you do not want to leave the choice up to the model. Two knobs:

- `tool_choice="any"`: force *some* tool, model picks which
- `tool_choice="<tool_name>"`: force a specific tool

In [ ]:
forced = model.bind_tools([add], tool_choice="add")
msg = forced.invoke("hi")  # we did not even ask for math; the model will be forced to call add
print(msg.tool_calls)

Use sparingly. A forced tool call is the model's way of saying "I had no choice." If your application *needs* a specific tool every call, that is often a sign you should not be using an agent at all, just call the function directly.

## Structured output: `with_structured_output`

Now switch problems. Instead of having the model call your tool, you want the *output itself* to be a typed object. Notebook 2 showed `JsonOutputParser` and `PydanticOutputParser`, which beg the model via the prompt to please return JSON. That works most of the time but fails on weird inputs.

`with_structured_output` is the more reliable way. Under the hood it uses the same tool-calling feature: it secretly defines a single tool that takes your schema as its argument, then forces the model to call that tool. The provider's structured-output guarantees apply, and you get a real Pydantic object back.

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with key facts."""
    title: str = Field(description="The official title")
    year: int = Field(description="Year of theatrical release")
    director: str = Field(description="Director's full name")
    rating: float = Field(description="IMDB rating, 0-10")

structured = model.with_structured_output(Movie)

movie = structured.invoke("Tell me about the movie Inception.")
print(type(movie).__name__)
print(movie)
print("director:", movie.director, "| rating:", movie.rating)

Notice we did not say a single word in our prompt about JSON or schemas. The model was *constrained* by the structured-output mode to fill in those fields. Compare this to the `PydanticOutputParser` example in notebook 2 where we manually injected format instructions. This is cleaner and more reliable.

## TypedDict version (no Pydantic dependency)

If you do not want to add Pydantic to your code, `TypedDict` works too and you get a plain `dict` back.

In [ ]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with key facts."""
    title: Annotated[str, ..., "The official title"]
    year: Annotated[int, ..., "Year of theatrical release"]
    director: Annotated[str, ..., "Director's full name"]

structured_dict = model.with_structured_output(MovieDict)
out = structured_dict.invoke("Tell me about The Matrix.")
print(type(out).__name__, out)

## `include_raw=True`: keep the raw message too

Sometimes you want both the parsed object *and* the original `AIMessage` (for token counts, ids, debugging). Pass `include_raw=True` and you get a dict back with both.

In [ ]:
structured_raw = model.with_structured_output(Movie, include_raw=True)
result = structured_raw.invoke("Tell me about The Godfather.")

print("keys:        ", list(result.keys()))
print("parsed:      ", result["parsed"])
print("raw type:    ", type(result["raw"]).__name__)
print("raw usage:   ", result["raw"].usage_metadata)
print("parsing err: ", result["parsing_error"])

**Practical tip**: in production code I almost always use `include_raw=True` even though I usually only consume `parsed`. The `raw` half lets me log token usage and debug parsing failures without re-running the call.

## Tool calling vs structured output: which when?

They use the same underlying mechanism, but mentally they solve different problems:

| You want... | Use |
|---|---|
| The model to *do* something (call a function, fetch data) and then talk again | Tool calling + the loop |
| The model's *final answer* to be a specific shape | `with_structured_output` |
| Both at once | Tool calling, where one of the tools returns your structured shape |

If your tool happens to return a Pydantic object, you can lean on the response_format Pydantic feature combined with `with_structured_output` to extract it cleanly. We will not go that deep here.

## Recap

You can now:

- define a tool with `@tool` (good docstring, real type hints)
- bind a list of tools onto a model with `bind_tools`
- read `response.tool_calls` and run the right function
- close the loop with a `ToolMessage` carrying the same `tool_call_id`
- generalize to multiple steps
- get reliable typed output via `with_structured_output(Pydantic)`

In notebook 4, we let `create_agent` write the loop for us. Now that you know what the loop does, you will recognize what `create_agent` is hiding.